# =========================
# INSTALLS (UNCHANGED)
# =========================

In [1]:

!pip install --no-index /kaggle/input/datasets/kurshidbasheer/biopython-offline/biopython-1.83-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
!pip install --no-index /kaggle/input/datasets/kurshidbasheer/pyg-2-7-torch-2-9-cpu-py312-kur/torch_geometric-2.7.0-py3-none-any.whl

Processing /kaggle/input/datasets/kurshidbasheer/biopython-offline/biopython-1.83-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
Processing /kaggle/input/datasets/kurshidbasheer/pyg-2-7-torch-2-9-cpu-py312-kur/torch_geometric-2.7.0-py3-none-any.whl


# =========================
# IMPORTS
# =========================

In [2]:

import torch, random
import numpy as np
import pandas as pd
import torch.nn as nn
from collections import defaultdict

from torch.utils.data import Dataset
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.utils import scatter

# =========================
# DEVICE
# =========================

In [3]:

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# =========================
# PATHS
# =========================

In [4]:

TRAIN_SEQ = "/kaggle/input/competitions/stanford-rna-3d-folding-2/train_sequences.csv"
TRAIN_LBL = "/kaggle/input/competitions/stanford-rna-3d-folding-2/train_labels.csv"
TEST_SEQ  = "/kaggle/input/competitions/stanford-rna-3d-folding-2/test_sequences.csv"

# =========================
# HYPERPARAMS
# =========================

In [5]:

WINDOW_SIZE = 400
STRIDE = 200
K_NEIGHBORS = 16

# =========================
# UTILS
# =========================

In [6]:

NUC_MAP = {'A':0,'U':1,'G':2,'C':3}

def clean_sequence(seq):
    return "".join([s for s in seq.upper() if s in NUC_MAP])

def one_hot(seq):
    x = torch.zeros(len(seq),4)
    for i,s in enumerate(seq):
        x[i,NUC_MAP[s]] = 1
    return x

# =========================
# GRAPH (FIXED)
# =========================

In [7]:

def build_graph(x, coords=None, k=16):
    L = x.size(0)

    row, col = [], []

    for i in range(L):
        local = list(range(max(0,i-k), min(L,i+k+1)))

        # 🔥 deterministic global edges
        step = max(1, L//k)
        global_nodes = list(range(0, L, step))

        neigh = list(set(local + global_nodes))
        if i in neigh: neigh.remove(i)

        row += [i]*len(neigh)
        col += neigh

    edge_index = torch.tensor([row,col], dtype=torch.long)

    rel = (edge_index[0]-edge_index[1]).float().unsqueeze(1)/L
    dist = rel.abs()
    edge_attr = torch.cat([dist, rel], dim=1)

    data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr)

    if coords is not None:
        data.y = coords

    # 🔥 POSITIONAL PRIOR (HELIX-LIKE)
    t = torch.linspace(0,1,L).unsqueeze(1)
    data.pos = torch.cat([
        t,
        torch.sin(2*np.pi*t),
        torch.cos(2*np.pi*t)
    ], dim=1)

    return data

# =========================
# DATASET
# =========================

In [8]:

class RNAWindowDataset(Dataset):
    def __init__(self, seq_csv, label_csv=None):
        self.df = pd.read_csv(seq_csv)
        self.has_labels = label_csv is not None

        self.seq_map = {
            row["target_id"]: clean_sequence(row["sequence"])
            for _,row in self.df.iterrows()
        }

        if self.has_labels:
            labels = pd.read_csv(label_csv, low_memory=False)
            labels["sid"] = labels["ID"].str.split("_").str[0]
            labels["idx"] = labels["ID"].str.split("_").str[1].astype(int)

            self.coords = {}
            for k,g in labels.groupby("sid"):
                g = g.sort_values("idx")
                xyz = torch.tensor(g[["x_1","y_1","z_1"]].values, dtype=torch.float32)

                valid = ~torch.isnan(xyz).any(dim=1)
                xyz = xyz[valid]

                if len(xyz)>0:
                    self.coords[k] = xyz

        self.samples = []

        for sid in self.df["target_id"]:
            seq = self.seq_map[sid]
            L = len(seq)

            if self.has_labels and sid in self.coords:
                L = min(L, self.coords[sid].shape[0])

            for s in range(0, L, STRIDE):
                e = min(s+WINDOW_SIZE, L)
                if e-s >= 20:
                    self.samples.append((sid,s,e))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sid,s,e = self.samples[idx]
        seq = self.seq_map[sid][s:e]

        coords = None
        if self.has_labels and sid in self.coords:
            coords = self.coords[sid][s:e]

        x = one_hot(seq)
        pos_feat = torch.arange(len(seq)).float().unsqueeze(-1)/len(seq)
        x = torch.cat([x,pos_feat],dim=1)

        return build_graph(x, coords)

# =========================
# KABSCH
# =========================

In [9]:

def kabsch(P, Q):
    Pc = P - P.mean(0,keepdim=True)
    Qc = Q - Q.mean(0,keepdim=True)

    C = Pc.t() @ Qc
    U,S,Vt = torch.linalg.svd(C)

    d = torch.det(U@Vt)
    D = torch.eye(3, device=P.device)
    D[-1,-1] = d

    R = U @ D @ Vt
    return Pc @ R

# =========================
# 🔥 HYBRID LOSS
# =========================

In [10]:
def hybrid_loss(pred, target, batch, epoch):
    loss = 0
    n = batch.max()+1

    for i in range(n):
        mask = batch==i
        P = pred[mask]
        Q = target[mask]

        P_aligned = kabsch(P, Q)

        # 🔥 normalize scale
        P_center = P_aligned - P_aligned.mean(0, keepdim=True)
        Q_center = Q - Q.mean(0, keepdim=True)

        scale = Q_center.std() + 1e-6

        Pn = P_center / scale
        Qn = Q_center / scale

        # --- MSE
        mse = ((Pn - Qn)**2).mean()

        # --- distance
        dist_P = torch.cdist(Pn, Pn)
        dist_Q = torch.cdist(Qn, Qn)
        dist_loss = ((dist_P - dist_Q)**2).mean()

        # --- TM
        L = P.shape[0]
        d0 = 1.24*(L-15)**(1/3)-1.8 if L>=30 else 0.5
        d = torch.norm(Pn-Qn, dim=1)
        tm = (1/(1+(d/d0)**2)).mean()

        w_tm = min(0.3, epoch/10)

        loss += (
            0.5*mse +
            0.15*dist_loss +
            w_tm*(1 - tm)
        )

    return loss/n

# =========================
# MODEL
# =========================

In [11]:

class EGNNLayer(nn.Module):
    def __init__(self, hidden):
        super().__init__()

        self.edge_mlp = nn.Sequential(
            nn.Linear(hidden*2+2, hidden),
            nn.SiLU(),
            nn.Linear(hidden, hidden)
        )

        self.node_mlp = nn.Sequential(
            nn.Linear(hidden*2, hidden),
            nn.SiLU(),
            nn.Linear(hidden, hidden)
        )

        self.coord_mlp = nn.Sequential(
            nn.Linear(hidden,1),
            nn.Tanh()
        )

    def forward(self,x,pos,edge_index,edge_attr):
        row,col = edge_index
        rel = pos[row]-pos[col]

        m = self.edge_mlp(torch.cat([x[row],x[col],edge_attr],dim=1))
        agg = scatter(m,row,dim=0,dim_size=x.size(0),reduce="mean")

        x = self.node_mlp(torch.cat([x,agg],dim=1))

        trans = self.coord_mlp(m)*rel
        delta = scatter(trans,row,dim=0,dim_size=pos.size(0),reduce="mean")

        pos = pos + 0.3 * delta
        return x,pos

class EGNNModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = nn.Linear(5,128)
        self.layers = nn.ModuleList([EGNNLayer(128) for _ in range(6)])
        self.dropout = nn.Dropout(0.1)

    def forward(self,data):
        x,pos = data.x,data.pos
        x = self.emb(x)

        for l in self.layers:
            x,pos = l(x,pos,data.edge_index,data.edge_attr)
            x = self.dropout(x)

        return pos

# =========================
# TRAIN
# =========================

In [12]:

def train_epoch(model, loader, opt, epoch):
    model.train()
    total=0

    for data in loader:
        data = data.to(DEVICE)
        opt.zero_grad()

        pred = model(data)

        loss = hybrid_loss(pred, data.y, data.batch, epoch)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()

        total += loss.item()

    return total/len(loader)

# =========================
# RUN
# =========================

In [13]:
from tqdm.auto import tqdm

def train_epoch(model, loader, opt, epoch):
    model.train()
    total = 0

    pbar = tqdm(loader, desc=f"Epoch {epoch}", leave=False)

    for data in pbar:
        data = data.to(DEVICE)
        opt.zero_grad()

        pred = model(data)
        loss = hybrid_loss(pred, data.y, data.batch, epoch)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()

        total += loss.item()

        pbar.set_postfix(loss=f"{loss.item():.4f}")

    return total / len(loader)


# =========================
# RUN
# =========================
train_ds = RNAWindowDataset(TRAIN_SEQ, TRAIN_LBL)
train_loader = DataLoader(train_ds, batch_size=4, shuffle=True)

model = EGNNModel().to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=5e-4)

for e in range(10):
    loss = train_epoch(model, train_loader, opt, e)
    print(f"Epoch {e} | loss = {loss:.6f}")

Epoch 0:   0%|          | 0/9873 [00:00<?, ?it/s]

Epoch 0 | loss = 0.862649


Epoch 1:   0%|          | 0/9873 [00:00<?, ?it/s]

Epoch 1 | loss = 0.862634


Epoch 2:   0%|          | 0/9873 [00:00<?, ?it/s]

...

Epoch 3 | loss = 0.862636


Epoch 4:   0%|          | 0/9873 [00:00<?, ?it/s]

Epoch 4 | loss = 0.879951


Epoch 5:   0%|          | 0/9873 [00:00<?, ?it/s]

KeyboardInterrupt: 